# Aula 13 - Notebook: Algoritmos de Busca (BFS e DFS) em Dígrafos Industriais

**Projeto:** SCADA-Core Automática — GRUPO 4  
**Unidade:** Planta Automatizada de Seleção e Classificação Óptica de Grãos com Separação Pneumática Tripla  
**Engenharia de Automação e Controle** — ECAA08  

---

### 1. Objetivos do Notebook
Neste notebook implementamos os algoritmos canônicos de travessia e busca topológica em grafos:
1. **Busca em Largura (BFS - *Breadth-First Search*):** Identificação de caminhos mínimos em saltos/estações de acionamento ($O(V + E)$), propagação de intertravamentos em cascata e árvore geradora BFS.
2. **Busca em Profundidade (DFS - *Depth-First Search*):** Rastreamento de rotas alternativas, classificação de arestas (Tree, Back, Forward, Cross Edges) e detecção inequívoca de ciclos fechados (circuito de reciclo de grãos).


### 2. Funções Auxiliares de Formatação Tabular ASCII


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


### 3. Modelo Estrutural da Planta de Grãos e Construtor da Rede


In [2]:
from typing import List, Dict, Any, Tuple, Optional
from collections import deque

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float,
                           tag_valvula: str, meio_transporte: str = "Duto", bitola: str = "DN200"):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Aresta": f"e{len(self.arestas_detalhes)+1}",
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador / ISA": tag_valvula,
            "Meio": meio_transporte,
            "Bitola": bitola
        })

    def obter_sucessores(self, v_nome: str) -> List[str]:
        u = self.v_to_idx[v_nome]
        return [self.idx_to_v[v] for v in range(self.n) if self.adj_binaria[u][v] == 1]

def criar_rede_padrao(incluir_reciclo: bool = True) -> GrafoTubulacao:
    nos = [
        "FN-101_Recepcao", "ALIM-101_Calha", "EST-201_Entrada", "BAL-301_Pesagem",
        "VIS-401_Inspecao", "DIV-501_Decisao", "EJE-602_EstacaoB", "EJE-603_EstacaoC",
        "CALHA-602_Secundario", "CALHA-603_Rejeito", "CALHA-701_Terminal",
        "SILO-701_Aprovados", "SILO-702_Secundario", "SILO-703_Rejeito"
    ]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("FN-101_Recepcao", "ALIM-101_Calha", 1.2, "XV-101", "Duto Vertical", "DN200")
    g.adicionar_tubulacao("ALIM-101_Calha", "EST-201_Entrada", 1.8, "c_ALIM", "Calha Vibratória", "L=300mm")
    g.adicionar_tubulacao("EST-201_Entrada", "BAL-301_Pesagem", 2.5, "c_EST/ST-201", "Correia", "L=400mm")
    g.adicionar_tubulacao("BAL-301_Pesagem", "VIS-401_Inspecao", 3.0, "WT-301/FT-301", "Correia", "L=400mm")
    g.adicionar_tubulacao("VIS-401_Inspecao", "DIV-501_Decisao", 1.0, "XS-401/KSA-401", "Zona Óptica", "L=400mm")
    g.adicionar_tubulacao("DIV-501_Decisao", "EJE-602_EstacaoB", 2.0, "ZC-602/p_POS602", "Correia", "L=400mm")
    g.adicionar_tubulacao("EJE-602_EstacaoB", "CALHA-602_Secundario", 0.8, "FY-602/ZSH-602", "Sopro B", "DN15")
    g.adicionar_tubulacao("EJE-602_EstacaoB", "EJE-603_EstacaoC", 1.5, "ST-201", "Correia", "L=400mm")
    g.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-603_Rejeito", 0.8, "FY-603/ZSH-601", "Sopro C", "DN15")
    g.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-701_Terminal", 2.2, "Descarga Inércia", "Correia", "L=400mm")
    g.adicionar_tubulacao("CALHA-701_Terminal", "SILO-701_Aprovados", 3.5, "XV-701", "Duto A", "DN250")
    g.adicionar_tubulacao("CALHA-602_Secundario", "SILO-702_Secundario", 4.0, "XV-702", "Duto B", "DN200")
    g.adicionar_tubulacao("CALHA-603_Rejeito", "SILO-703_Rejeito", 3.2, "XV-703", "Duto C", "DN200")
    if incluir_reciclo:
        g.adicionar_tubulacao("SILO-702_Secundario", "FN-101_Recepcao", 12.0, "XV-104", "Elevador Reciclo", "Canecas 6pol")
    return g

rede = criar_rede_padrao(incluir_reciclo=True)


### 4. Implementação do Motor de Busca Topológica: Classe `BuscadorTopologico`


In [3]:
class BuscadorTopologico:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def busca_largura_bfs(self, origem: str) -> Tuple[Dict[str, int], Dict[str, Optional[str]], List[str]]:
        """Executa BFS a partir de origem. Retorna (distancias em saltos, antecessores, ordem de visita)."""
        dist = {v: -1 for v in self.g.vertices}
        pai = {v: None for v in self.g.vertices}
        ordem_visita = []
        
        fila = deque([origem])
        dist[origem] = 0
        
        while fila:
            u = fila.popleft()
            ordem_visita.append(u)
            for v in self.g.obter_sucessores(u):
                if dist[v] == -1:  # Nao visitado
                    dist[v] = dist[u] + 1
                    pai[v] = u
                    fila.append(v)
        return dist, pai, ordem_visita

    def reconstruir_caminho_bfs(self, origem: str, destino: str, pai: Dict[str, Optional[str]]) -> Optional[List[str]]:
        """Reconstroi o caminho minimo em saltos entre origem e destino."""
        if origem == destino:
            return [origem]
        if pai[destino] is None:
            return None
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            if atual == origem:
                break
            atual = pai[atual]
        return list(reversed(caminho))

    def busca_profundidade_dfs(self) -> Dict[str, Any]:
        """Executa DFS completa classificando arestas e detectando ciclos."""
        # Estados: 0 = BRANCO (nao visitado), 1 = CINZA (em exploracao na pilha), 2 = PRETO (finalizado)
        cor = {v: 0 for v in self.g.vertices}
        d = {v: 0 for v in self.g.vertices}
        f = {v: 0 for v in self.g.vertices}
        pai = {v: None for v in self.g.vertices}
        tempo = [0]
        arestas_classificadas = []
        ciclos_detectados = []

        def dfs_visit(u):
            tempo[0] += 1
            d[u] = tempo[0]
            cor[u] = 1
            for v in self.g.obter_sucessores(u):
                if cor[v] == 0:  # Tree Edge
                    arestas_classificadas.append({"Origem": u, "Destino": v, "Classificacao": "Tree Edge (Arvore)"})
                    pai[v] = u
                    dfs_visit(v)
                elif cor[v] == 1:  # Back Edge -> Ciclo
                    arestas_classificadas.append({"Origem": u, "Destino": v, "Classificacao": "Back Edge (Retorno - CICLO)"})
                    ciclos_detectados.append((u, v))
                elif cor[v] == 2:  # Forward ou Cross Edge
                    if d[u] < d[v]:
                        arestas_classificadas.append({"Origem": u, "Destino": v, "Classificacao": "Forward Edge (Avanco)"})
                    else:
                        arestas_classificadas.append({"Origem": u, "Destino": v, "Classificacao": "Cross Edge (Cruzamento)"})
            cor[u] = 2
            tempo[0] += 1
            f[u] = tempo[0]

        for v in self.g.vertices:
            if cor[v] == 0:
                dfs_visit(v)

        return {
            "d": d,
            "f": f,
            "pai": pai,
            "arestas": arestas_classificadas,
            "ciclos": ciclos_detectados,
            "possui_ciclo": len(ciclos_detectados) > 0
        }


### 5. Execução do BFS: Caminhos Mínimos em Saltos e Árvore de Despacho


In [4]:
buscador = BuscadorTopologico(rede)
dist_bfs, pai_bfs, ordem_bfs = buscador.busca_largura_bfs("FN-101_Recepcao")

dados_bfs = []
for v in rede.vertices:
    caminho = buscador.reconstruir_caminho_bfs("FN-101_Recepcao", v, pai_bfs)
    caminho_str = " -> ".join([p.split("_")[0] for p in caminho]) if caminho else "Inacessível"
    dados_bfs.append({
        "Equipamento / Nó": v,
        "Saltos (h)": dist_bfs[v],
        "Antecessor Direto": pai_bfs[v] if pai_bfs[v] else "---",
        "Caminho Topológico Mínimo": caminho_str
    })

print("=== RELATÓRIO DE BUSCA EM LARGURA (BFS) A PARTIR DE FN-101 ===")
print(formatar_tabela(dados_bfs))

print("\nOrdem de Exploração por Camadas:")
print(" -> ".join(ordem_bfs))


=== RELATÓRIO DE BUSCA EM LARGURA (BFS) A PARTIR DE FN-101 ===
Equipamento / Nó     | Saltos (h) | Antecessor Direto    | Caminho Topológico Mínimo                                                                                    
---------------------+------------+----------------------+--------------------------------------------------------------------------------------------------------------
FN-101_Recepcao      | 0          | ---                  | FN-101                                                                                                       
ALIM-101_Calha       | 1          | FN-101_Recepcao      | FN-101 -> ALIM-101                                                                                           
EST-201_Entrada      | 2          | ALIM-101_Calha       | FN-101 -> ALIM-101 -> EST-201                                                                                
BAL-301_Pesagem      | 3          | EST-201_Entrada      | FN-101 -> ALIM-101 -> EST-201 -> 

### 6. Execução do DFS: Classificação de Arestas e Detecção de Ciclos no Reciclo


In [5]:
res_dfs = buscador.busca_profundidade_dfs()

dados_dfs_nos = []
for v in rede.vertices:
    dados_dfs_nos.append({
        "Equipamento / Nó": v,
        "Descoberta d[v]": res_dfs["d"][v],
        "Finalização f[v]": res_dfs["f"][v],
        "Pai DFS": res_dfs["pai"][v] if res_dfs["pai"][v] else "RAIZ"
    })

print("=== RELATÓRIO DE TEMPOS DE VARREDURA DFS (TEOREMA DO PARÊNTESE) ===")
print(formatar_tabela(dados_dfs_nos))

print("\n=== CLASSIFICAÇÃO TOPOLÓGICA DAS ARESTAS (DFS) ===")
print(formatar_tabela(res_dfs["arestas"]))

print(f"\nPresença de Ciclos Fechados: {res_dfs['possui_ciclo']}")
if res_dfs["possui_ciclo"]:
    for u, v in res_dfs["ciclos"]:
        print(f"-> Ciclo Confirmado pela Back Edge: {u} ===> {v} (Circuito de Reciclo e Reteste)")


=== RELATÓRIO DE TEMPOS DE VARREDURA DFS (TEOREMA DO PARÊNTESE) ===
Equipamento / Nó     | Descoberta d[v] | Finalização f[v] | Pai DFS             
---------------------+-----------------+------------------+---------------------
FN-101_Recepcao      | 1               | 28               | RAIZ                
ALIM-101_Calha       | 2               | 27               | FN-101_Recepcao     
EST-201_Entrada      | 3               | 26               | ALIM-101_Calha      
BAL-301_Pesagem      | 4               | 25               | EST-201_Entrada     
VIS-401_Inspecao     | 5               | 24               | BAL-301_Pesagem     
DIV-501_Decisao      | 6               | 23               | VIS-401_Inspecao    
EJE-602_EstacaoB     | 7               | 22               | DIV-501_Decisao     
EJE-603_EstacaoC     | 8               | 17               | EJE-602_EstacaoB    
CALHA-602_Secundario | 18              | 21               | EJE-602_EstacaoB    
CALHA-603_Rejeito    | 9               | 